In [ ]:
# Current Philharmonia parser: HTTP-only, no browser/runtime packages required.
import importlib.util
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("PhilharmoniaParser")


def _load_parser_module():
    roots = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root / "philharmonia_parser.py"]
        try:
            candidates.extend(sorted(root.rglob("philharmonia_parser.py")))
        except Exception:
            pass
        for candidate in candidates:
            if not candidate.exists():
                continue
            spec = importlib.util.spec_from_file_location("philharmonia_parser", candidate)
            if spec and spec.loader:
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                logger.info("loaded parser from %s", candidate)
                return module
    raise FileNotFoundError("philharmonia_parser.py not found in Kaggle kernel inputs")


def _status(event, *, phase, status, **progress):
    try:
        if "kaggle_status_update" in globals():
            kaggle_status_update(phase=phase, **progress)
        if "kaggle_status_event" in globals():
            kaggle_status_event(event, phase=phase, status=status, progress=progress)
    except Exception as exc:
        logger.warning("Kaggle status callback failed: %s", exc)


_status("parse_started", phase="parse", status="running", progress_percent=10, progress_label="загрузка афиши")
parser = _load_parser_module()
events = parser.fetch_philharmonia_events()
output_path = Path("/kaggle/working/philharmonia_results.json")
output_path.write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8")
logger.info("wrote %d events to %s", len(events), output_path)
_status(
    "source_report_written",
    phase="report",
    status="complete",
    events_done=len(events),
    events_total=len(events),
    progress_percent=95,
    progress_label=f"события {len(events)}/{len(events)}",
)
